# D2D MOBO Debug Notebook

This notebook is a research/debug interface for the package implementation. The
authoritative Step 2B run is `mobo_kit.d2d_step2b_debug.run_d2d_step2b_debug`,
which reads the completed workbook without modifying it and watermarks every
candidate artifact as **DEBUG ONLY - NOT APPROVED FOR EXPERIMENT**.

## Outline
- 0. Setup and portable paths
- 1. Read the completed R0 workbook
- 2. Normalize inputs
- 2.5. Calculate and validate D2D scores
- 3. Fit GP models
- 4-5. Optional model diagnostics
- 6. Run the guarded Step 2B debug adapter
- 7-9. Review the adapter's watermarked outputs

## 0. Setup & Imports


**Imports** Bring in libraries for data handling, modeling, and plotting.  
If CUDA isn't available, the CPU path still works for a workshop-scale demo.


In [ ]:
# --- Imports & setup ---
%load_ext autoreload
%autoreload 2

# Core
import os, sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

parent_dir = os.path.dirname(os.getcwd())
if parent_dir not in sys.path:
    sys.path.insert(0, parent_dir)

# Torch / BoTorch / GPyTorch
import torch
from mobo_kit.utils import csv_to_config, split_XY, get_objective_names, np_to_torch
from mobo_kit.design import build_design_from_config

from mobo_kit.lhs import lhs_dataframe, lhs_dataframe_optimized
from mobo_kit.constraints import constraints_from_config

from mobo_kit.plotting import plot_distribution, plot_correlation_heatmap, plot_PCA

from mobo_kit.data import x_normalizer_np

from mobo_kit.models import fit_gp_models
from mobo_kit.constraints import check_clausius_clapeyron_np
from mobo_kit.lhs import lhs_dataframe
#from mobo_kit.plotting import plot_pareto, plot_hypervolume_trajectory

# if torch.cuda.is_available():
#     device = torch.device("cuda")
# elif torch.backends.mps.is_available():
#     device = torch.device("mps")
# else:
device = torch.device("cpu")

print(f"Using device: {device}")


## Portable workbook, configuration, and debug-output paths

In [ ]:
# Supply private inputs explicitly through environment variables; no private path is tracked.
import os

cwd = Path.cwd().resolve()
repo_root = cwd.parent if cwd.name == "notebooks" else cwd
private_workbook = os.environ.get("MOBO_KIT_D2D_PRIVATE_WORKBOOK")
private_config = os.environ.get("MOBO_KIT_D2D_PRIVATE_CONFIG")
if not private_workbook or not private_config:
    raise RuntimeError(
        "Set MOBO_KIT_D2D_PRIVATE_WORKBOOK and MOBO_KIT_D2D_PRIVATE_CONFIG "
        "to explicit ignored local files before running campaign cells."
    )
workbook_path = Path(private_workbook).expanduser().resolve()
config_path = Path(private_config).expanduser().resolve()
save_path = repo_root / "local_outputs" / "d2d_step2b_debug" / "notebook_seed73"
diagnostics_path = repo_root / "local_outputs" / "d2d_step2b_debug" / "notebook_diagnostics_seed73"
diagnostics_path.mkdir(parents=True, exist_ok=True)

print("Workbook:", workbook_path)
print("Config:", config_path)
print("Notebook diagnostics:", diagnostics_path)
print("Debug bundle (created only when enabled):", save_path)

## 1. Read the completed R0 workbook

The Step 2B reader selects the 15 numeric `Sample number` rows from the
35-column v3 workbook. Inputs are mapped from B:K, the final objectives are
selected by their exact Z/AA/AB headers, and Stability plus AD:AI are excluded.
The source workbook is opened read-only and is never saved.

### Completed R0 data

Control identities and any observed-only off-grid exceptions are supplied only
by the ignored private configuration. They are retained for GP training and
distance diagnostics, while every new candidate remains on the configured grid.

In [ ]:
from mobo_kit.d2d_campaign import (
    D2D_INPUT_COLUMNS,
    D2D_OBJECTIVE_COLUMNS,
    load_d2d_debug_config,
    load_d2d_workbook_frame,
    prepare_d2d_training_data,
)

resolved_config = load_d2d_debug_config(config_path)
design = resolved_config.design
data_rows, workbook_audit = load_d2d_workbook_frame(
    workbook_path,
    expected_profile=resolved_config.workbook_profile,
    expected_sample_ids=resolved_config.expected_sample_ids,
    allowed_input_exceptions=resolved_config.off_grid_exceptions,
)
training_data = prepare_d2d_training_data(data_rows, resolved_config)

X_df = pd.DataFrame(training_data.X_phys_all, columns=D2D_INPUT_COLUMNS)
Y_df = pd.DataFrame(training_data.Y_objectives, columns=D2D_OBJECTIVE_COLUMNS)
obj_names = list(D2D_OBJECTIVE_COLUMNS)

print("Workbook profile:", workbook_audit.profile)
print("Rows:", len(data_rows), "Inputs:", X_df.shape, "Objectives:", Y_df.shape)
display(X_df.head())
display(Y_df.head())

### Optional R0 LHS reference (not used by Step 2B)

The completed workbook already contains R0 observations, so the guarded debug
path does not regenerate LHS points. Set the flag below only for a separate,
input-only design exercise.

In [ ]:
run_optional_lhs = False
row_constraints = []
max_abs_corr = 0.32
lhs_batch_size = 14

if run_optional_lhs:
    lhs_df = lhs_dataframe_optimized(
        design,
        n=lhs_batch_size,
        seed=42,
        row_constraints=row_constraints,
        max_abs_corr=max_abs_corr,
        verbose=True,
    )
    display(lhs_df)
else:
    lhs_df = pd.DataFrame(columns=design.names)
    print("Skipping LHS because completed R0 data are available.")

In [ ]:
import numpy as np
import pandas as pd

# 1. Pull the pre-calculated choices from the Design object
# 'var_array' was created by build_design using the 'make_linspace' function
summary_data = []
for i, name in enumerate(design.names):
    # .size gives the number of valid grid points for this parameter
    n_choices = design.var_array[i].size 
    summary_data.append({
        "Parameter": name,
        "Choices": n_choices,
        "Min": design.lowers[i],
        "Max": design.uppers[i],
        "Step": design.steps[i]
    })

# 2. Display the table
space_summary_df = pd.DataFrame(summary_data)
display(space_summary_df)

# 3. Calculate Total Conditions (Product of all choices)
total_conditions = np.prod(space_summary_df["Choices"].values)
print(f"\nTOTAL UNIQUE CONDITIONS IN SEARCH SPACE: {total_conditions:,.0f}")

In [ ]:
if run_optional_lhs:
    round_worklist_path = diagnostics_path / "Round_0_Input_Only_Reference.csv"
    round_worklist_path.parent.mkdir(parents=True, exist_ok=True)
    input_reference = lhs_df.copy()
    for objective in D2D_OBJECTIVE_COLUMNS:
        input_reference[objective] = ""
    input_reference.to_csv(round_worklist_path, index=False)
    print("Input-only reference saved to:", round_worklist_path)

### Visualize data distribution


#### Standardize inputs to visualize correlation (NE addition 260407)

In [ ]:
if run_optional_lhs:
    scaler = StandardScaler()
    lhs_standardized = scaler.fit_transform(lhs_df)
    lhs_df_std = pd.DataFrame(lhs_standardized, columns=lhs_df.columns)
else:
    lhs_df_std = pd.DataFrame(columns=design.names)

In [ ]:
if run_optional_lhs:
    _ = plot_distribution(lhs_df, title="LHS Feature Distributions", save=diagnostics_path / "lhs_dist.png")
    _ = plot_correlation_heatmap(lhs_df_std, title="LHS Pearson Correlation", save=diagnostics_path / "lhs_corr.png")
    _ = plot_PCA(lhs_df_std, title="LHS PCA (2D)", save=diagnostics_path / "lhs_pca.png")

## 2. Normalize Data and Create Tensors/Arrays


**Inputs.** Scale to [0, 1] per dimension. 

This helps GP hyperparameters learn sensibly. There are utility functions to normalize and standardize the outputs, but this is currently handled internally in `fit_gp_models`.

**Tensors.** Some Botorch functions use tensor objects rather than arrays and vice versa. It's good to have both!


In [ ]:
# Inputs -> fixed [0, 1] bounds from the configured design.
X_np = X_df.to_numpy(dtype=float)
X_norm = x_normalizer_np(X_np, design)
X_t, _ = np_to_torch(X_norm, device=device, return_device=True)
print("Normalized input shape:", X_t.shape)

## 2.5 Calculate and Validate D2D Scores

The final BO objectives are Excel Z `Uniformity score`, AA
`Optoelectronic score`, and AB `Thickness score`. All three are maximized and
used directly; they are not clipped to `[0, 1]`. Experimental analysis owns the
final scores, while the support equations below are validation only. The known
uniformity inconsistency is a debug warning, and Stability is ignored.

- Uniformity support: `Coverage * (1 - Uniformity) * Phase purity`
- Optoelectronic support: `log10(P * Q)`
- Thickness support: `exp(-((mean(valid T1:T4) - 650.0) / 250.0) ** 2)`

The thickness equation has **no factor 0.5**, uses the unrounded valid T1:T4
mean, and excludes `T anom`.

In [ ]:
from mobo_kit.d2d_scores import (
    compute_thickness_average,
    compute_thickness_score,
    validate_supplied_d2d_scores,
)

def normalize_missing_thickness_cell(value):
    # Convert blank/whitespace/NBSP thickness cells to NaN.
    if value is None:
        return np.nan
    if isinstance(value, str) and not value.replace("\u00a0", " ").strip():
        return np.nan
    return value

thickness_columns = ["T1", "T2", "T3", "T4"]
normalized_thickness = data_rows[thickness_columns].map(
    normalize_missing_thickness_cell
)

In [ ]:
# Package helper implements exp(-((mean_t - 650.0) / 250.0) ** 2).
thickness_means = normalized_thickness.apply(
    lambda row: compute_thickness_average(*row.tolist()), axis=1
)
calculated_thickness_scores = normalized_thickness.apply(
    lambda row: compute_thickness_score(row.tolist(), target=650.0, scale=250.0),
    axis=1,
)
display(
    pd.DataFrame(
        {
            "Sample number": data_rows["Sample number"],
            "unrounded T1:T4 mean": thickness_means,
            "calculated thickness score": calculated_thickness_scores,
            "supplied Thickness score": data_rows["Thickness score"],
        }
    )
)

In [ ]:
score_validation = validate_supplied_d2d_scores(data_rows)
display(score_validation.frame)
print("Uniformity warnings:", score_validation.uniformity_warning_count)
score_validation.raise_for_errors()  # fatal for missing, opto, or thickness errors

In [ ]:
objective_columns = [
    "Uniformity score",
    "Optoelectronic score",
    "Thickness score",
]
Y = data_rows[objective_columns].astype(float)
Y_df = Y.copy()
Y_np = Y.to_numpy(dtype=float)
(X_t, Y_t), _ = np_to_torch(X_norm, Y_np, device=device, return_device=True)
obj_names = objective_columns
print("Tensor shapes:", X_t.shape, Y_t.shape)

## 3. Fit Gaussian Process (GP) Models


We fit one **Gaussian Process (GP)** for each authoritative final score: Excel Z
`Uniformity score`, AA `Optoelectronic score`, and AB `Thickness score`. The
three models preserve the direct score scales and the resolved optimization
direction is **max/max/max**. This gives an uncertainty-aware model of how each
final score varies with the process parameters.

- **Defaults:**  
  - Kernel: `MaternKernel(nu=2.5, ard_num_dims=d)` (smooth, ARD per feature)  
  - Likelihood: `GaussianLikelihood` with an inferred homoskedastic noise level  
    (includes a small positive floor to prevent overfitting)

- **Optional overrides:**  
  - You may pass your own **kernel function(s)** (e.g., RBF, Matern with different ν, etc.)  
  - You may pass **noise priors** (e.g., LogNormal or Gamma) to guide the noise estimate.

This example shows passing custom priors and a kernel to demonstrate how it works, but you can omit them entirely to use the defaults.


In [ ]:
from mobo_kit.models import fit_gp_models

# The model internally standardizes each output but returns posterior values on
# the original Z/AA/AB scales.
model = fit_gp_models(X_t, Y_t)

## 4. LOOCV Model Selection (optional)

Instead of hand-choosing kernel functions and noise priors, you can let the repo 
**automatically compare candidates** using **Leave-One-Out Cross-Validation (LOOCV)**.

- The function `loocv_select_models` tries multiple `(kernel × noise)` combinations.
- By default, it uses:
  - **Kernel options:** RBF, Matern(ν=0.5), Matern(ν=1.5), Matern(ν=2.5)  
  - **Noise priors:** `None` (free noise level) and `LogNormal(-4.0, 0.5)`
- For each fold (leave one point out), it re-fits the GP, predicts the held-out point, 
  and records **R²** and **RMSE**.  
- After sweeping all combinations, it selects the best configuration per objective 
  and re-fits on the full dataset.

**Pros:** 
- removes guesswork.
- gives metrics for each option.  

**Cons:** 
- expensive when you have many data points (since it re-fits N×(#kernels×#priors) times).
- poor fits when handling very small and noisy datasets and have convergence errors.

You can skip this step if you are happy with the defaults from Step 3.


In [ ]:
# Optional LOOCV model selection. Disabled by default because it refits many GPs.
run_loocv = False
if run_loocv:
    from mobo_kit.models import loocv_select_models
    model_cv, results_df = loocv_select_models(
        X_t,
        Y_t,
        objective_names=obj_names, # list of objective column names
        device=X_t.device,  # optional; inferred from X_t if omitted
    )
    display(results_df.sort_values(["objective", "rmse"]))
else:
    results_df = pd.DataFrame()
    model_cv = None
    print("Skipping optional LOOCV. Set run_loocv = True to enable it.")

### Display LOOCV Results

In [ ]:
# Display fitted model details from the default model (in source code).
for i, gp in enumerate(model.models):
    print(f"--- Model {i+1} ({obj_names[i]}) ---")
    kernel = gp.covar_module.base_kernel
    print("Kernel:", type(kernel).__name__)
    if hasattr(kernel, "nu"):
        print("Matern nu:", kernel.nu)
    prior_type = type(getattr(gp.likelihood.noise_covar, "noise_prior", None)).__name__
    print("Noise prior:", prior_type)
    print("Lengthscales:", gp.covar_module.base_kernel.lengthscale.detach().cpu().numpy().flatten())
    print("Outputscale:", gp.covar_module.outputscale.item())
    print("Noise:", gp.likelihood.noise.item())
    print()

## 5. Posterior Predictions & Diagnostics


With a fitted GP model (`model` from Step 3 or 4), we can now evaluate how well it explains the data.  
This step does two things:

1. **Posterior predictions:**  
   - Use `posterior_report(model, X_t)` to get the **predicted mean** and **uncertainty (std)** for each objective at the training points.  
   - These predictions are automatically converted back into the original units (because the model internally standardizes outputs).

2. **Diagnostics:**  
   - `plot_parity_np` shows **predicted vs. true values** for each objective, with optional error bars from the GP’s predictive uncertainty.  
   - `plot_shap` estimates **feature importance** (mean absolute SHAP values per input dimension), so you can see which process parameters most influence each objective.  
   - `compute_metrics` calculates **R²** and **RMSE**, and can also return per-point residuals and z-scores to help check model fit quality.

In [ ]:
from mobo_kit.models import posterior_report
from mobo_kit.plotting import plot_parity_np, plot_shap
from mobo_kit.metrics import compute_metrics
pred_mean, pred_std = posterior_report(model, X_t)
fig_train, metrics_train = plot_parity_np(
    Y_np,
    pred_mean,
    pred_std,
    objective_names=obj_names,
    save=str(diagnostics_path / "parity_train.png"),
)
run_shap = False
if run_shap:
    # plot_shap calls the fitted GP posterior directly, so explain the same
    # normalized input space used to train the model.
    fig_shap = plot_shap(
        design,
        X_norm,
        model,
        objective_names=obj_names,
        save=str(diagnostics_path / "shap.png"),
    )
else:
    fig_shap = None
    print("Skipping SHAP by default to keep memory usage low. Set run_shap = True to enable it.")
metrics_df = compute_metrics(
    true_Y=Y_np,
    pred_mean=pred_mean,
    pred_std=pred_std,
    objective_names=obj_names,
    add_residuals=True,
    add_zscores=True,
)
display(metrics_df)

## 6. R1 UCB-HVI Debug Candidate Generation

The guarded Step 2B path ranks grid-valid conditions with **R1 UCB-HVI** and
shared local penalization. The optimization contract is Excel Z `Uniformity
score`, AA `Optoelectronic score`, and AB `Thickness score`, with
**max/max/max** directions and the fixed reference point
`[-0.01, -10.0, -0.01]`. The reference point is a campaign constant; it is not
derived from the observed worst values.

The adapter selects exactly five unique conditions, expands each condition to
three replicate rows, and watermarks every artifact **DEBUG ONLY - NOT APPROVED
FOR EXPERIMENT**. The batch is for algorithm diagnostics, not experimental
execution.

### Pareto front and hypervolume diagnostics
- The **Pareto front** consists of non-dominated points: no other point is strictly better across *all* objectives.  
- The **hypervolume** measures the space dominated by the Pareto front relative to the fixed campaign reference point.  
- As we add better candidates, the hypervolume increases.

In [ ]:
from botorch.utils.multi_objective.pareto import is_non_dominated
from mobo_kit.d2d_campaign import (
    D2D_REFERENCE_POINT_UTILITY,
    build_d2d_objective_transform,
)
from mobo_kit.metrics import compute_ref_pareto_hv

directions = ["max", "max", "max"]
objective_transform = build_d2d_objective_transform()
Y_t_transformed = objective_transform(Y_t)  # identity; values remain Z/AA/AB
transformed_ref_point_t = torch.as_tensor(
    D2D_REFERENCE_POINT_UTILITY,
    dtype=Y_t.dtype,
    device=Y_t.device,
)
if not torch.all(Y_t_transformed > transformed_ref_point_t):
    raise ValueError("Every observed score must strictly dominate the fixed reference point.")

_, pareto_Y_t_transformed, hv_val = compute_ref_pareto_hv(
    Y_t_transformed,
    D2D_REFERENCE_POINT_UTILITY.copy(),
)
pareto_mask = is_non_dominated(Y_t_transformed)
pareto_Y_t_real_world = Y_t[pareto_mask]
print("Fixed reference point:", transformed_ref_point_t.cpu().numpy())
print("Observed hypervolume:", hv_val)
print("Pareto count:", pareto_Y_t_real_world.shape[0])

### Guarded R1 algorithm-debug adapter

The package adapter calls the Step 2A discrete pool, UCB-HVI, and shared local
penalization APIs directly. It selects five unique conditions and expands them
to three physical replicates only after acquisition. It never invokes the
legacy raw-objective proposal path and never writes to Excel.

In [ ]:
from mobo_kit.d2d_step2b_debug import run_d2d_step2b_debug

# Deliberate opt-in: candidate files are always watermarked debug-only.
run_campaign_debug = False
debug_result = None
if run_campaign_debug:
    debug_result = run_d2d_step2b_debug(
        workbook_path,
        config_path,
        save_path,
        overwrite=False,
        run_sensitivity=True,
    )
    display(debug_result.candidates_unique)
else:
    print("Debug proposal not run. Set run_campaign_debug=True to create the guarded bundle.")

### Legacy global-distance sandbox retired

The earlier experimental cells used APIs and constraints that are not part of
the active package. The Step 2A shared normalized-distance selector and the
Step 2B adapter above are authoritative.

In [ ]:
# Retired legacy sandbox cell. See run_d2d_step2b_debug above.
pass

In [ ]:
# Retired legacy sandbox cell. See run_d2d_step2b_debug above.
pass

In [ ]:
# Retired legacy sandbox cell. See run_d2d_step2b_debug above.
pass

In [ ]:
# Retired legacy sandbox cell. See run_d2d_step2b_debug above.
pass

### Review the guarded adapter result

In [ ]:
if debug_result is not None:
    X_next_df = debug_result.candidates_unique.copy()
    display(X_next_df)
else:
    X_next_df = pd.DataFrame()

In [ ]:
if debug_result is not None:
    print("Debug artifacts:", debug_result.output_dir)
    print("Unique conditions:", len(debug_result.candidates_unique))
    print("Replicate rows:", len(debug_result.replicate_worklist))

## 7. Review R1 UCB-HVI Debug Diagnostics

When `debug_result` is available, review the guarded adapter outputs directly:

1. `debug_result.candidates_unique` contains five grid-valid conditions in
   physical units, plus predicted means and standard deviations for Z
   `Uniformity score`, AA `Optoelectronic score`, and AB `Thickness score`.
2. `base_ucb_hvi`, `penalty_factor`, and `penalized_log_score` document the R1
   UCB-HVI ranking and local-penalization effect. These values are R1 algorithm
   diagnostics from the guarded adapter.
3. `debug_result.model_diagnostics` and the saved sensitivity summary expose
   model-fit and batch-robustness checks.

The objectives remain direct-score **max/max/max**, the reference point remains
fixed at `[-0.01, -10.0, -0.01]`, and every displayed or saved candidate is
**DEBUG ONLY - NOT APPROVED FOR EXPERIMENT**.



In [ ]:
if debug_result is not None:
    display(debug_result.candidates_unique)
    display(debug_result.model_diagnostics)
else:
    print("Run the guarded adapter to review predictions and diagnostics.")

## 8. Visualize Your Progress (Work in Progress...)

We track learning progress by plotting the **hypervolume (HV)** after each batch of **observed** results.  
As you collect new data and the Pareto front improves, HV should **monotonically increase** (or stay flat).

**Key points**
- Use **observed outcomes** (not predictions).  
- Keep the campaign reference point fixed at `[-0.01, -10.0, -0.01]`; do not recompute it from observed minima or worst values.  
- Compute HV on the **cumulative dataset** up through each batch.

In [ ]:
print(f"Current observed hypervolume: {hv_val:.6f}")
if debug_result is not None:
    print("Candidate and sensitivity diagnostics are stored in the guarded bundle.")

This plot shows the **current Pareto front** (green) in 3D objective space and an optional **predicted next batch**.  

The **reference point** is drawn as a red star.

In [ ]:
# All three displayed axes are higher-is-better final scores.
directions = ["max", "max", "max"]
print("Pareto objectives:", list(D2D_OBJECTIVE_COLUMNS))
print("Reference point:", D2D_REFERENCE_POINT_UTILITY.tolist())

## 9. Save Outputs


Only the guarded adapter writes Step 2B outputs, under `local_outputs`. It never
modifies the source workbook, and every candidate artifact remains **DEBUG ONLY
- NOT APPROVED FOR EXPERIMENT**.


In [ ]:
if debug_result is not None:
    print("The adapter has already saved the watermarked candidate and replicate CSV files.")

In [ ]:
if debug_result is not None:
    display(debug_result.replicate_worklist)
else:
    print("No outputs written; the debug adapter remains opt-in.")